# Electricity consumption

Fetch and plot your recent half-hourly electricity consumption for a meter using `squidink`.

You will need your Octopus Energy **API key**, your **MPAN**, and your **meter serial number** (all on your Octopus dashboard).

Credentials are loaded from a local `.env` file (see `.env.example`).

In [ ]:
import os
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from dotenv import load_dotenv

from squidink import Client, Granularity

In [ ]:
load_dotenv()  # load OCTOPUS_* from a local .env

api_key = os.environ["OCTOPUS_API_KEY"]
mpan = os.environ["OCTOPUS_MPAN"]
serial = os.environ["OCTOPUS_METER_SERIAL_NUMBER"]

In [ ]:
# The last 7 days, up to now (Europe/London).
london_tz = ZoneInfo("Europe/London")
period_to = datetime.now(london_tz)
period_from = period_to - timedelta(days=7)

with Client(api_key=api_key) as client:
    readings = client.get_consumption_readings(
        mpan, serial, period_from=period_from, period_to=period_to
    )

print(f"{len(readings)} readings from {period_from:%Y-%m-%d} to {period_to:%Y-%m-%d}")

In [ ]:
import plotly.graph_objects as go

# Each reading covers a 30-minute interval; draw a bar that wide, anchored at its start.
half_hour_ms = 30 * 60 * 1000
fig = go.Figure(
    go.Bar(
        x=[r.interval_start for r in readings],
        y=[r.consumption for r in readings],
        width=half_hour_ms,
        offset=0,
    )
)
fig.update_layout(
    title=f"Electricity Consumption: {serial}",
    xaxis_title="Time",
    yaxis_title="Consumption (kWh)",
)
fig.show()

In [ ]:
# Average daily consumption per month over the last 24 months, via the pandas-series helper.
# period_to = midnight on the 1st of this month; period_from = the 1st, two years earlier.
period_to = datetime.now(london_tz).replace(day=1, hour=0, minute=0, second=0, microsecond=0)
period_from = period_to.replace(year=period_to.year - 2)

with Client(api_key=api_key) as client:
    monthly = client.get_consumption_series(
        mpan,
        serial,
        granularity=Granularity.MONTH,
        period_from=period_from,
        period_to=period_to,
    )

# The Series is already Europe/London-aware, so divide each month's total by its days.
average_daily = monthly / monthly.index.days_in_month

fig = go.Figure(go.Bar(x=average_daily.index, y=average_daily.values))
fig.update_layout(
    title=f"Average Daily Consumption by Month: {serial}",
    xaxis_title="Month",
    yaxis_title="Average Daily Consumption (kWh)",
)
fig.show()